# The Transformer Model

## Setup

### Install packages

In [ ]:
!pip install -q datasets evaluate transformers torch

### Import libraries

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.nn.functional import softmax
import tqdm as notebook_tqdm

### Instantiate model

In [ ]:
checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"      # model for sentiment analysis
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, output_hidden_states=True)

## Model workflow

![transf sequence.png](./images/Transformer%20sequence.png)

### Tokenization
Transformer models only accept tensors as input, so text needs to be pre-processed by a tokenizer:
1. Split the input text into words, subwords, or symbols (like punctuation), called tokens
2. Map each token to an integer, using the downloaded dictionary
3. Include additional inputs required by the model

In [ ]:
sequences = ["There's a plethora of content to learn at Cisco Live.", "I'm bored, when can I leave?"]
tokens = tokenizer(sequences, padding=True, truncation=True, return_tensors="pt")

print(tokens)

* Every tensor needs to have the same length, so shorter tensors (for shorter sentences) will need padding.
* The attention mask signals what positions inside the tensor are meaningful (marked with '1').

### Inference
![transf architecture-3.png](./images/Transformer%20architecture.png)
The embeddings layer converts tokens included inside input vectors into tensors that represent each token. Subsequent layers manipulate those tensors using the attention mechanism to produce the final representation of the sentences. Given some inputs, the model outputs what we call _hidden states_ (aka _features_), high-dimensional tensors representing the contextual understanding of that input by the Transformer model. These hidden states are usually inputs to another part of the model, known as the _head_.

In [ ]:
outputs = model(**tokens)

print(outputs.hidden_states[-1].size())     # number of sentences, sequence size, vector dimensions
print(outputs.hidden_states[-1])            # hidden state for the last layer (2 sentences)

The Transformer model output is sent directly to the model _head_ to be processed. Different heads are designed around tackling a specific task. The model heads take the high-dimensional tensor of hidden states as input and project them onto a different dimension. 

For example, sentiment analysis requires a head for sequence classification. It will process those input tensors and output 2 values (_logits_): probability for negative and positive sentiments. 

In [ ]:
logits = outputs.logits

print(logits.shape)
print(logits)


However the model output comes in the shape of logits, which are raw unnormalized scores that need to go through a `softmax` (normalization) + loss function.

### Visualizing attention

The `softmax` function normalizes logits to values between 0 and 1, so we can use those to define the probabilities for negative (first position or index 0) or positive (second position or index 1).

In [ ]:
print(model.config.id2label)

probabilities = softmax(logits, dim=1)

print(probabilities[0].tolist())
predicted_class_id_0 = logits[0].argmax().item()
print(model.config.id2label[predicted_class_id_0])

print(probabilities[1].tolist())
predicted_class_id_1 = logits[1].argmax().item()
print(model.config.id2label[predicted_class_id_1])